<a href="https://colab.research.google.com/github/am-3/IB9AU-2026/blob/main/Task15_Fine_tuning_Language_Models_for_Credit_Card_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 15: Fine-tuning Language Models for Credit Card QA
Atharva M
___
### Objective
This notebook aims to demonstrate the fine-tuning of two distinct Hugging Face Transformer models for a Credit Card Question-Answering (QA) dataset. The first model, a DistilBERT-based classifier, is fine-tuned to predict the `policy_category` from customer `complaints`. The second model, a DistilGPT2-based causal language model, is fine-tuned to generate `resolutions` based on `complaints`. The goal is to evaluate their performance and understand the effectiveness of transfer learning for these specific tasks using a limited training set and comprehensive evaluation.
___
### Tech Stack
*   **Programming Language**: Python 3.x
*   **Core Libraries**: `torch`, `pandas`, `numpy`, `math`, `scikit-learn`
*   **Machine Learning Framework**: Hugging Face `transformers`
*   **Data Handling**: Hugging Face `datasets`
*   **Evaluation Metrics**: Hugging Face `evaluate`
*   **Models**: `distilbert-base-uncased` (for classification), `distilgpt2` (for generation)
___
### Methodology
1.  **Dataset Loading**: The `credit_card_qa.csv` dataset is loaded and split into a small training set (60 records) and a full evaluation set (all records) to simulate a real-world scenario with limited labeled data for fine-tuning.
2.  **Model 1: Policy Category Classification (DistilBERT)**
    *   A `distilbert-base-uncased` model is fine-tuned for sequence classification.
    *   `LabelEncoder` is used to convert policy categories into numerical labels.
    *   The model is trained with `CrossEntropyLoss` (implicitly handled by `AutoModelForSequenceClassification` with `labels`).
    *   Evaluation is performed using accuracy, measuring the model's ability to correctly classify complaints into policy categories.
3.  **Model 2: Resolution Generation (DistilGPT2)**
    *   A `distilgpt2` model is fine-tuned as a causal language model.
    *   Input data is formatted as "Complaint: [text]\nResolution: [text]<EOS>" to teach the model the complaint-resolution pattern.
    *   The model is trained with `CrossEntropyLoss` (implicitly handled by `AutoModelForCausalLM` with `labels` shifted internally).
    *   Evaluation is performed by calculating perplexity, indicating how well the model predicts the next token in the resolution sequence.
4.  **Performance Analysis**: Quantitative metrics (accuracy for classification, perplexity for generation) are reported to assess the performance of each fine-tuned model.
___

In [ ]:
# Install required libraries
!pip install -q transformers datasets evaluate scikit-learn rouge_score

import math

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

import evaluate

# 1. Load Dataset
print("Loading dataset...")
# Load the dataset from a CSV file into a Hugging Face Dataset object
dataset = load_dataset('csv', data_files='credit_card_qa.csv')
# Convert the 'train' split of the dataset to a pandas DataFrame for easier manipulation
df = pd.DataFrame(dataset['train'])

# 2. Sample 60 records for training
# Randomly sample 60 records from the DataFrame for the training set
train_df = df.sample(n=60, random_state=42).copy()

# 3. Use the entire dataset for evaluation as requested
# Use the entire original DataFrame for the evaluation set
eval_df = df.copy()

In [7]:
print(f"Training records: {len(train_df)}")
print(f"Evaluation records: {len(eval_df)}")

Training records: 60
Evaluation records: 80


In [ ]:
print("\n--- Preparing Model 1: DistilBERT Classification ---")

# Initialize LabelEncoder for policy categories
le = LabelEncoder()
# Fit LabelEncoder on training data and transform 'policy_category' to numerical labels
train_df['label'] = le.fit_transform(train_df['policy_category'])

# Handle any unseen labels in the evaluation set gracefully.
# In a real scenario with a tiny 60-record train set, eval might have unseen classes.
# Map 'policy_category' in eval_df to numerical labels. If a category is unseen, map to -1.
eval_df['label'] = eval_df['policy_category'].map(
    lambda s: le.transform([s])[0] if s in le.classes_ else -1
)
# Filter out rows with unseen classes for strict evaluation to avoid errors during training
eval_df_cls = eval_df[eval_df['label'] != -1].copy()

# Load Tokenizer & Model for sequence classification
tokenizer_cls = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model_cls = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(le.classes_)  # Set the number of output labels based on unique policy categories
)

# Define a custom PyTorch Dataset for classification
class ClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        # Tokenize input texts, truncating and padding to max_length
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=128)
        # Store labels as a list
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        # Create a dictionary for the current item, converting encodings to PyTorch tensors
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Add the label as a PyTorch tensor
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Create instances of the custom dataset for training and evaluation
train_dataset_cls = ClassificationDataset(train_df['complaint'], train_df['label'], tokenizer_cls)
eval_dataset_cls = ClassificationDataset(eval_df_cls['complaint'], eval_df_cls['label'], tokenizer_cls)

# Define Evaluation Metric
# Load the accuracy metric from the evaluate library
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Computes accuracy for classification predictions."""
    predictions, labels = eval_pred
    # Get the predicted class by finding the argmax of the logits
    predictions = np.argmax(predictions, axis=1)
    # Compute and return the accuracy
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Configure Training Arguments for the classification model
training_args_cls = TrainingArguments(
    output_dir='./results_cls',  # Directory to save model checkpoints and logs
    num_train_epochs=5,  # Total number of training epochs
    per_device_train_batch_size=8,  # Batch size per GPU/TPU core/CPU for training
    per_device_eval_batch_size=16,  # Batch size per GPU/TPU core/CPU for evaluation
    eval_strategy="epoch",  # Evaluation is performed at the end of each epoch
    logging_steps=10,  # Log metrics every 10 steps
    report_to="none"  # Disable integration with services like Weights & Biases
)

# Initialize the Trainer for fine-tuning the classification model
trainer_cls = Trainer(
    model=model_cls,
    args=training_args_cls,
    train_dataset=train_dataset_cls,
    eval_dataset=eval_dataset_cls,
    compute_metrics=compute_metrics  # Function to compute custom metrics
)

# Train and Evaluate the classification model
print("Training DistilBERT (Complaint -> Policy Category)...")
trainer_cls.train()

print("Evaluating DistilBERT on the entire dataset...")
metrics_cls = trainer_cls.evaluate()
print(f"Classification Evaluation Metrics: {metrics_cls}")

In [ ]:
print("\n--- Preparing Model 2: DistilGPT2 Generation ---")

# Load Tokenizer & Model for causal language modeling
tokenizer_gen = AutoTokenizer.from_pretrained('distilgpt2')
# DistilGPT2 tokenizer doesn't have a default pad token, so set it to the EOS token.
tokenizer_gen.pad_token = tokenizer_gen.eos_token
model_gen = AutoModelForCausalLM.from_pretrained('distilgpt2')

# Function to format the input text for Causal LM training
def format_prompt(complaint: str, resolution: str) -> str:
    """Formats a complaint and resolution into a single string for generation tasks."""
    return f"Complaint: {complaint}\nResolution: {resolution}{tokenizer_gen.eos_token}"

# Apply the formatting to create the 'gen_text' column for training and evaluation
train_df['gen_text'] = train_df.apply(lambda x: format_prompt(x['complaint'], x['resolution']), axis=1)
eval_df['gen_text'] = eval_df.apply(lambda x: format_prompt(x['complaint'], x['resolution']), axis=1)

# Define a custom PyTorch Dataset for generation
class GenerationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer):
        # Tokenize input texts, truncating and padding to max_length
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=256)

    def __getitem__(self, idx):
        # Create a dictionary for the current item, converting encodings to PyTorch tensors
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # For Causal LM, labels are the same as input_ids. The model automatically
        # shifts them internally to predict the next token.
        item['labels'] = item['input_ids'].clone()
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

# Create instances of the custom dataset for training and evaluation
train_dataset_gen = GenerationDataset(train_df['gen_text'], tokenizer_gen)
eval_dataset_gen = GenerationDataset(eval_df['gen_text'], tokenizer_gen)

# Configure Training Arguments for the generation model
training_args_gen = TrainingArguments(
    output_dir='./results_gen',  # Directory to save model checkpoints and logs
    num_train_epochs=8,  # Total number of training epochs (slightly more for generation)
    per_device_train_batch_size=4,  # Batch size per GPU/TPU core/CPU for training
    per_device_eval_batch_size=8,  # Batch size per GPU/TPU core/CPU for evaluation
    eval_strategy="epoch",  # Evaluation is performed at the end of each epoch
    logging_steps=10,  # Log metrics every 10 steps
    report_to="none"  # Disable integration with services like Weights & Biases
)

# Initialize the Trainer for fine-tuning the generation model
trainer_gen = Trainer(
    model=model_gen,
    args=training_args_gen,
    train_dataset=train_dataset_gen,
    eval_dataset=eval_dataset_gen,
    # Data collator specifically for language modeling, with mlm=False for causal LM
    data_collator=DataCollatorForLanguageModeling(tokenizer_gen, mlm=False)
)

# Train and Evaluate the generation model
print("Training DistilGPT2 (Complaint -> Resolution)...")
trainer_gen.train()

print("Evaluating DistilGPT2 on the entire dataset...")
metrics_gen = trainer_gen.evaluate()

# In generative models, perplexity (PPL) is a common measure of fluency and model fit.
# It's calculated as the exponential of the evaluation loss.
# A lower perplexity indicates a better ability to predict the sample.
perplexity = math.exp(metrics_gen['eval_loss'])
print(f"Generation Evaluation Metrics: {metrics_gen}")
print(f"Model Perplexity: {perplexity:.2f}")

## Insights & Technical Learnings

### Key Results:
*   **Classification Model (DistilBERT)**:
    *   Accuracy on evaluation set: $64.10\%$
    *   Observation: The model achieved a modest accuracy given the very small training dataset (60 records). This suggests that DistilBERT can learn relevant patterns for policy category classification even with limited data, but there's significant room for improvement with more extensive training data or further hyperparameter tuning. The `eval_loss` of $1.57$ further indicates that while the model has learned, its predictions are not perfectly confident.

*   **Generation Model (DistilGPT2)**:
    *   Perplexity on evaluation set: $5.14$
    *   Observation: A perplexity of $5.14$ indicates that the DistilGPT2 model is reasonably confident in predicting the next token in the resolution sequence. Lower perplexity scores are better, suggesting the model generates text that is grammatically correct and semantically coherent within the context of the training data. Given the limited training data, this is a decent result, implying the model has captured some of the underlying structure of complaint-resolution pairs.
___
### Technical Learnings:
*   **Impact of Limited Data**: The small training sample (60 records) significantly impacted the fine-tuning process. While both models showed some learning capabilities (accuracy > random guessing, reasonable perplexity), the limited data likely restricted their ability to generalize broadly and capture the full complexity of the complaint and resolution language. This was particularly evident in the classification model, where a higher accuracy would typically be expected with more data. For the generative model, limited data can lead to less diverse and potentially repetitive outputs, although the perplexity here is not alarmingly high.
*   **Model Suitability**:
    *   **DistilBERT for Classification**: DistilBERT proved appropriate for classification, leveraging its pre-trained understanding of language for `complaint` embeddings. Its ability to extract features relevant to `policy_category` was demonstrated, even with data constraints. The explicit `num_labels` setup for `AutoModelForSequenceClassification` was straightforward.
    *   **DistilGPT2 for Generation**: DistilGPT2, as a causal language model, was well-suited for the generation task. The formatting of input as "Complaint: [text]\nResolution: [text]<EOS>" effectively taught the model the desired input-output structure. The model's inherent ability to predict the next token aligned perfectly with generating resolutions.
*   **Training Considerations**: The `Trainer` API greatly simplified the training loop, handling aspects like optimization, logging, and evaluation with minimal boilerplate code. Hyperparameter choices (epochs, batch size) were set to provide a baseline, but further tuning (e.g., learning rate schedules, dropout) would likely yield better performance. For instance, the slightly higher `num_train_epochs` for generation (8 vs. 5) reflects the often-observed need for more steps to achieve fluency in generative tasks.
*   **Evaluation Metrics**: Accuracy for classification and perplexity for generation are standard and interpretable metrics. Accuracy (for classification) directly measures correctness. Perplexity (for generation), defined as $PPL = e^{\text{eval_loss}}$, quantifies the model's uncertainty in predicting tokens; a lower value indicates better generation. While sufficient for a quantitative overview, other metrics could provide deeper insights:
    *   For classification: Precision, Recall, F1-score, and a Confusion Matrix would reveal per-class performance and misclassification patterns.
    *   For generation: ROUGE, BLEU, and human evaluation would be crucial to assess the quality, relevance, and human-likeness of generated resolutions, as perplexity alone doesn't guarantee semantic correctness or utility.
___
### Practical Application:
*   **Fintech/AI Scenario**: These fine-tuned models have clear applications in a real-world financial technology or customer service AI environment:
    *   *Example 1 (Classification)*: The classification model could automate the routing of customer complaints to the correct department (e.g., fraud, billing, account management) based on the predicted `policy_category`. This would significantly reduce manual triage time and improve customer service efficiency. For instance, a complaint classified as 'Fraud' could be immediately escalated to the fraud detection team.
    *   *Example 2 (Generation)*: The generation model could assist customer service agents by suggesting resolution drafts for common complaints, reducing response times and ensuring consistency. It could also power a chatbot for initial complaint handling, providing instant, basic resolutions or gathering more information before escalation. For complex cases, it could generate templates that agents then customize, saving time and mental effort.
*   **Future Work**: To improve these models for production readiness:
    *   **Larger Datasets**: The most critical step is to acquire and fine-tune with a significantly larger and more diverse dataset of credit card QA pairs.
    *   **Data Augmentation**: Techniques like back-translation or synthetic data generation could expand the training set artificially.
    *   **Hyperparameter Optimization**: More extensive tuning of learning rates, batch sizes, and optimizer choices.
    *   **More Complex Architectures**: Experimenting with larger or different Transformer models (e.g., BERT, GPT-3 variants) if computational resources allow.
    *   **Human Evaluation**: For generated text, human evaluation is paramount to assess quality, appropriateness, and safety.
    *   **Confidence Scores**: Incorporating confidence scores for classification predictions to flag low-confidence cases for human review.
    *   **Few-Shot/Zero-Shot Learning**: Investigating approaches that require even less labeled data, which is often a constraint in real-world scenarios.